# Tugas #4: Data-Centric Improvement and Model Interpretation

Notebook ini melanjutkan pipeline terbaik Tugas #3. Model dan preprocessing **tidak diubah**:
yang diubah hanya data train (imbalance treatment, review label Cleanlab), supaya selisih
performa bisa diatributkan ke perubahan data.

Setup yang dipertahankan dari Tugas #3:
- Split raw 70/30 stratified (`random_state=42`). Semua CV hanya di data train.
- `StratifiedKFold(5, shuffle=True, random_state=42)`, fold identik di semua eksperimen.
- Primary metric: **MCC**. Macro F1 ditambahkan sebagai kolom pendamping.
- Test set baru dipakai sekali di evaluasi akhir.

In [1]:
import sys, os, warnings
sys.path.append('../src')
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd

from data_utils import load_data
from preprocessing import split_raw
from models import get_models
from pipeline_experiments import build_pipeline, CHI2_K
from data_centric import cv_evaluate, get_cv, PRIMARY_METRIC

pd.set_option('display.width', 140)
TABLE_DIR, FIG_DIR = '../results/tables', '../results/figures'
os.makedirs(TABLE_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

df = load_data()
X_train_raw, X_test_raw, y_train, y_test = split_raw(df)
X, y = X_train_raw, y_train   # alias: semua CV di bawah HANYA memakai data train

print('Train:', X_train_raw.shape, '| proporsi positif:', round(y_train.mean(), 4))
print('Test :', X_test_raw.shape, '| proporsi positif:', round(y_test.mean(), 4), '(tidak disentuh)')

Train: (8631, 17) | proporsi positif: 0.1548
Test : (3699, 17) | proporsi positif: 0.1546 (tidak disentuh)


## 1. Baseline (D0)

D0 adalah pipeline terbaik Tugas #3 apa adanya: `CorrectedPreprocessor` (one-hot `Month` dan
`VisitorType`, chi-square top-20, standard scaling) lalu SMOTE 1:1 di dalam pipeline, lalu
XGBoost `max_depth=2`. Di Tugas #3 pipeline ini tetap yang terbaik di test set (MCC 0,5955),
mengalahkan versi log-transform/pruning dan versi tuned.

Sebagai sanity check, MCC CV harus sama dengan Tugas #3 (0,6257).

In [2]:
xgb = get_models()['XGBoost']
pipe_d0 = build_pipeline(xgb, k=CHI2_K, drop_features=None, log_transform=False, use_smote=True)

summary_d0, folds_d0 = cv_evaluate(pipe_d0, X, y)
print('D0 (baseline Tugas #3), 5-fold CV di data train:')
print(summary_d0)

T3_MCC = 0.6257
assert abs(summary_d0.loc['MCC', 'Mean'] - T3_MCC) < 1e-4, 'Baseline tidak cocok dengan Tugas #3'
print(f'\nSanity check OK: MCC = {summary_d0.loc["MCC", "Mean"]:.4f} (Tugas #3: {T3_MCC})')

D0 (baseline Tugas #3), 5-fold CV di data train:
                        Mean     Std
Accuracy              0.8940  0.0069
Precision             0.6343  0.0217
TPR (Recall)          0.7470  0.0276
F1-Score              0.6857  0.0190
TNR                   0.9209  0.0075
MCC                   0.6257  0.0228
auROC                 0.9265  0.0076
auPR                  0.7307  0.0233
Precision (weighted)  0.9029  0.0060
F1-Score (weighted)   0.8974  0.0064
Macro F1              0.8110  0.0115

Sanity check OK: MCC = 0.6257 (Tugas #3: 0.6257)


In [3]:
print('Skor per fold D0:')
print(folds_d0[['MCC', 'Macro F1', 'F1-Score', 'TPR (Recall)', 'Precision', 'auPR']].round(4))

summary_d0.to_csv(f'{TABLE_DIR}/tugas4_d0_baseline_cv.csv')
folds_d0.round(4).to_csv(f'{TABLE_DIR}/tugas4_d0_baseline_folds.csv')

Skor per fold D0:
           MCC  Macro F1  F1-Score  TPR (Recall)  Precision    auPR
Fold 1  0.6147    0.8055    0.6769        0.7388     0.6246  0.7032
Fold 2  0.6707    0.8338    0.7232        0.7828     0.6720  0.7601
Fold 3  0.6165    0.8048    0.6777        0.7640     0.6090  0.7479
Fold 4  0.6191    0.8073    0.6803        0.7491     0.6231  0.7390
Fold 5  0.6076    0.8033    0.6703        0.7004     0.6426  0.7035


Baseline cocok dengan Tugas #3, jadi fold dan pipeline terbukti identik. Std MCC
antar fold sekitar 0,02: selisih antar eksperimen data di bawah ukuran ini perlu dibaca sebagai
noise, bukan perbaikan.

## 2. Identifikasi Masalah Kualitas Data

EDA lengkap tidak diulang. Bagian ini mengambil temuan Tugas #2/#3 yang belum dituntaskan,
lalu menambah beberapa pengecekan yang belum pernah dilakukan. Semua pengecekan hanya di data train.

In [4]:
d = X.copy(); d['Revenue'] = y.values
feat = list(X.columns)

# (a) Class imbalance
print('(a) Distribusi kelas train:', y.value_counts().to_dict(), f'| positif = {y.mean():.2%}')

# (b) Duplikat: apakah ada pasangan fitur identik dengan label berbeda?
dup_mask = d.duplicated(subset=feat, keep=False)
grp = d[dup_mask].groupby(feat, dropna=False)['Revenue'].nunique()
print(f'(b) Baris dalam grup duplikat: {dup_mask.sum()} ({d.duplicated(subset=feat).sum()} salinan ekstra), '
      f'{len(grp)} grup, grup dengan label konflik: {(grp > 1).sum()}, '
      f'proporsi Revenue=1 di duplikat: {d[dup_mask].Revenue.mean():.2%}')
print('    Profil median duplikat:',
      d[dup_mask][['ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues']]
      .median().to_dict())

(a) Distribusi kelas train: {0: 7295, 1: 1336} | positif = 15.48%
(b) Baris dalam grup duplikat: 126 (74 salinan ekstra), 52 grup, grup dengan label konflik: 0, proporsi Revenue=1 di duplikat: 0.00%
    Profil median duplikat: {'ProductRelated': 1.0, 'ProductRelated_Duration': 0.0, 'BounceRates': 0.2, 'ExitRates': 0.2, 'PageValues': 0.0}


In [5]:
# (c) Nilai inkonsisten: jumlah halaman > 0 tetapi durasi = 0 (dan kebalikannya)
rows = []
for cnt, dur in [('Administrative', 'Administrative_Duration'),
                 ('Informational', 'Informational_Duration'),
                 ('ProductRelated', 'ProductRelated_Duration')]:
    m = (d[cnt] > 0) & (d[dur] == 0)
    rows.append({'pasangan': f'{cnt} > 0, {dur} = 0', 'n': int(m.sum()),
                 'Revenue=1 (%)': round(100 * d[m].Revenue.mean(), 2)})
    rows.append({'pasangan': f'{cnt} = 0, {dur} > 0', 'n': int(((d[cnt] == 0) & (d[dur] > 0)).sum()),
                 'Revenue=1 (%)': None})
rows.append({'pasangan': 'Durasi negatif (semua kolom durasi)',
             'n': int((d[[c for c in feat if c.endswith('_Duration')]] < 0).any(axis=1).sum()),
             'Revenue=1 (%)': None})
rows.append({'pasangan': 'BounceRates > ExitRates', 'n': int((d.BounceRates > d.ExitRates).sum()),
             'Revenue=1 (%)': round(100 * d[d.BounceRates > d.ExitRates].Revenue.mean(), 2)})
print('(c) Pengecekan konsistensi:')
print(pd.DataFrame(rows).to_string(index=False))

# (d) Segmen anomali: sesi "bounce murni" (BounceRates = ExitRates = 0,2, nilai maksimum)
bb = (d.BounceRates == 0.2) & (d.ExitRates == 0.2)
td = d[[c for c in feat if c.endswith('_Duration')]].sum(axis=1)
print(f'\n(d) Sesi BounceRates=ExitRates=0,2: {bb.sum()} baris, Revenue=1: {d[bb].Revenue.sum()} '
      f'| sesi total durasi = 0: {(td == 0).sum()} baris, Revenue=1: {d[td == 0].Revenue.sum()}')
print('    Sesi bounce murni tapi Revenue=1 (kandidat label mencurigakan, dibawa ke Cleanlab):')
print(d[bb & (d.Revenue == 1)][['ProductRelated', 'ProductRelated_Duration', 'PageValues',
                                'Month', 'VisitorType', 'TrafficType']])

(c) Pengecekan konsistensi:
                                       pasangan   n  Revenue=1 (%)
Administrative > 0, Administrative_Duration = 0  82          24.39
Administrative = 0, Administrative_Duration > 0   0            NaN
  Informational > 0, Informational_Duration = 0 160          21.25
  Informational = 0, Informational_Duration > 0   0            NaN
ProductRelated > 0, ProductRelated_Duration = 0 509           1.18
ProductRelated = 0, ProductRelated_Duration > 0   0            NaN
            Durasi negatif (semua kolom durasi)   0            NaN
                        BounceRates > ExitRates 132          10.61

(d) Sesi BounceRates=ExitRates=0,2: 492 baris, Revenue=1: 2 | sesi total durasi = 0: 507 baris, Revenue=1: 2
    Sesi bounce murni tapi Revenue=1 (kandidat label mencurigakan, dibawa ke Cleanlab):
      ProductRelated  ProductRelated_Duration  PageValues Month        VisitorType  TrafficType
2524               3                      0.0         0.0   Nov  Returning_

In [6]:
# (e) PageValues: fitur paling dominan di EDA, sekaligus kandidat leakage-adjacent
pv = d.PageValues > 0
print(f'(e) PageValues > 0: {pv.sum()} sesi, Revenue=1 rate {d[pv].Revenue.mean():.2%} '
      f'| PageValues = 0: Revenue=1 rate {d[~pv].Revenue.mean():.2%}')
print(f'    Revenue=1 dengan PageValues = 0: {((~pv) & (d.Revenue == 1)).sum()} dari {d.Revenue.sum()} '
      f'({((~pv) & (d.Revenue == 1)).sum() / d.Revenue.sum():.1%})')

# (f) Level kategori langka (segmen bermasalah untuk one-hot / generalisasi)
for c in ['Browser', 'OperatingSystems', 'TrafficType', 'VisitorType']:
    vc = d[c].value_counts()
    print(f'(f) {c}: {(vc < 30).sum()} level dengan < 30 baris (total {vc[vc < 30].sum()} baris)')
print('    Month di train (Jan dan Apr tidak ada di dataset):', d.Month.value_counts().to_dict())

(e) PageValues > 0: 1911 sesi, Revenue=1 rate 57.04% | PageValues = 0: Revenue=1 rate 3.66%
    Revenue=1 dengan PageValues = 0: 246 dari 1336 (18.4%)
(f) Browser: 3 level dengan < 30 baris (total 13 baris)
(f) OperatingSystems: 3 level dengan < 30 baris (total 23 baris)
(f) TrafficType: 7 level dengan < 30 baris (total 63 baris)
(f) VisitorType: 0 level dengan < 30 baris (total 0 baris)
    Month di train (Jan dan Apr tidak ada di dataset): {'May': 2377, 'Nov': 2066, 'Mar': 1351, 'Dec': 1208, 'Oct': 381, 'Sep': 319, 'Aug': 308, 'Jul': 299, 'June': 189, 'Feb': 133}


### Ringkasan masalah dan tindakan

| # | Masalah | Bukti (data train) | Status di Tugas #2/#3 | Tindakan Tugas #4 |
|---|---|---|---|---|
| a | Class imbalance | Positif 15,48% (1.336 dari 8.631) | Ditangani dengan SMOTE 1:1, tidak pernah dibandingkan dengan alternatif | **Step 3**: bandingkan tanpa treatment, `scale_pos_weight`, SMOTE |
| b | Duplikat | 126 baris dalam 52 grup (74 salinan ekstra). Semua Revenue=0, tanpa label konflik. Profilnya sesi bounce 1 halaman, durasi 0 | Dicatat di EDA, tidak ditangani | **Tidak dihapus** (dilarang brief, dan wajar: sesi bounce identik memang bisa terjadi dari user berbeda). Efeknya ke CV dicek di Step 7 |
| c | Durasi 0 padahal jumlah halaman > 0 | 509 sesi (ProductRelated), 160 (Informational), 82 (Administrative). Tidak ada kebalikannya, tidak ada nilai negatif | Belum dicek | Konsisten dengan cara Google Analytics menghitung durasi (halaman terakhir sesi tidak punya durasi), jadi bukan error. **Tidak dikoreksi** |
| d | Sesi bounce murni dengan Revenue=1 | 492 sesi BounceRates=ExitRates=0,2, hanya 2 yang Revenue=1 (tanpa durasi, PageValues 0) | Belum dicek | Kandidat label mencurigakan, **dibawa ke Step 4 (Cleanlab)** |
| e | PageValues sangat dominan | Rate pembelian 57% jika > 0, 3,7% jika = 0. Korelasi 0,49 dengan target | Dipakai apa adanya | Definisinya dihitung dari halaman yang dikunjungi *sebelum transaksi*, jadi berpotensi leakage-adjacent. **Diinvestigasi di Step 6/7** dengan SHAP |
| f | Level kategori langka | TrafficType 7 level < 30 baris, Browser 3, OS 3, VisitorType 'Other' 67 | One-hot hanya untuk Month/VisitorType, kode lain dipakai sebagai angka | Dicatat sebagai segmen berisiko, dicek lagi kalau muncul di misklasifikasi |
| g | Penulisan kategori tidak seragam | `Month` memakai 'June' (4 huruf) sementara bulan lain 3 huruf ('Jul', 'Aug') | Belum dicek | Kosmetik saja: one-hot memperlakukannya sebagai satu level yang konsisten. **Tidak diubah** |

Missing value tidak ada (sudah diverifikasi di Tugas #2), jadi tidak dibahas ulang. Outlier IQR
(misalnya `ProductRelated_Duration`) sudah dibahas di Tugas #2/#3 dan tidak relevan untuk model
pohon, jadi tidak diinvestigasi ulang.

In [7]:
# Simpan tabel masalah kualitas data (termasuk yang TIDAK diubah) untuk laporan
issues = pd.DataFrame([
    ['a', 'Class imbalance', 'Positif 15,48% (1.336 dari 8.631)', 'Ditindaklanjuti', 'Step 3: none vs scale_pos_weight vs SMOTE'],
    ['b', 'Duplikat', '126 baris, 52 grup, 74 salinan ekstra; semua Revenue=0, tanpa label konflik; profil sesi bounce 1 halaman',
     'Tidak diubah', 'Dilarang dihapus dan wajar (sesi bounce identik dari user berbeda); efek ke CV dicek di Step 7'],
    ['c', 'Durasi 0 padahal jumlah halaman > 0', 'ProductRelated 509, Informational 160, Administrative 82; tidak ada kebalikan/negatif',
     'Tidak diubah', 'Konsisten dengan perhitungan durasi Google Analytics (halaman terakhir tanpa durasi)'],
    ['d', 'Sesi bounce murni dengan Revenue=1', '2 dari 492 sesi Bounce=Exit=0,2 (durasi 0, PageValues 0)',
     'Ditindaklanjuti', 'Kandidat label mencurigakan, diverifikasi di Step 4 (Cleanlab)'],
    ['e', 'PageValues sangat dominan', 'Rate beli 57,0% jika > 0 vs 3,7% jika = 0; korelasi 0,49',
     'Ditindaklanjuti', 'Potensi leakage-adjacent, diinvestigasi di Step 6/7 (SHAP)'],
    ['f', 'Level kategori langka', 'TrafficType 7 level < 30 baris (63), Browser 3 (13), OS 3 (23)',
     'Tidak diubah', 'Dicatat sebagai segmen berisiko, dicek bila muncul di misklasifikasi'],
    ['g', 'Penulisan kategori tidak seragam', "Month 'June' vs singkatan 3 huruf", 'Tidak diubah', 'Kosmetik, one-hot tetap konsisten'],
], columns=['id', 'masalah', 'bukti_train', 'status', 'tindakan'])
issues.to_csv(f'{TABLE_DIR}/tugas4_data_quality_issues.csv', index=False)
issues[['id', 'masalah', 'status']]

,id,masalah,status
0,a,Class imbalance,Ditindaklanjuti
1,b,Duplikat,Tidak diubah
2,c,Durasi 0 padahal jumlah halaman > 0,Tidak diubah
3,d,Sesi bounce murni dengan Revenue=1,Ditindaklanjuti
4,e,PageValues sangat dominan,Ditindaklanjuti
5,f,Level kategori langka,Tidak diubah
6,g,Penulisan kategori tidak seragam,Tidak diubah


## 3. Class Imbalance: Tanpa Treatment vs Class Weighting vs SMOTE

Tiga varian dibandingkan dengan preprocessing, model (XGBoost `max_depth=2`), dan fold yang sama:

- **I0, tanpa treatment**: pipeline D0 tanpa langkah SMOTE.
- **I1, class weighting**: XGBoost tidak punya `class_weight` (parameter itu diam-diam diabaikan),
  jadi padanannya `scale_pos_weight = n_negatif / n_positif`. Rasio dihitung dari data train;
  karena fold stratified, rasio di setiap fold train praktis sama (dicek di bawah).
- **I2, SMOTE 1:1** (= D0). SMOTE berada di dalam pipeline, jadi hanya diterapkan ke fold train.
  Fold validasi tetap pada distribusi asli.

Metrik yang dibaca: MCC (primary), Macro F1, F1/Precision/Recall kelas positif, dan auPR.
auPR tidak bergantung threshold, jadi dipakai untuk membedakan "model jadi lebih baik memisahkan kelas"
dari "threshold efektifnya hanya bergeser".

In [8]:
from data_centric import neg_pos_ratio, summary_row

spw = neg_pos_ratio(y)
fold_ratios = [neg_pos_ratio(y.iloc[tr]) for tr, _ in get_cv().split(X, y)]
print(f'scale_pos_weight (train): {spw:.4f} | per fold train: {[round(r, 4) for r in fold_ratios]}')

imb_pipes = {
    'I0 Tanpa treatment':        build_pipeline(xgb, use_smote=False),
    'I1 Class weighting (spw)':  build_pipeline(xgb, use_smote=False, use_class_weight=True, scale_pos_weight=spw),
    'I2 SMOTE 1:1 (= D0)':       build_pipeline(xgb, use_smote=True),
}
imb_summary, imb_folds = {}, {}
for name, pipe in imb_pipes.items():
    imb_summary[name], imb_folds[name] = cv_evaluate(pipe, X, y)

# Sanity check: I2 harus identik dengan D0
assert imb_summary['I2 SMOTE 1:1 (= D0)'].equals(summary_d0)

COLS = ['MCC', 'Macro F1', 'F1-Score', 'Precision', 'TPR (Recall)', 'TNR', 'auPR', 'auROC', 'Accuracy']
imb_table = pd.DataFrame([summary_row(s.loc[COLS], n) for n, s in imb_summary.items()])
imb_table.to_csv(f'{TABLE_DIR}/tugas4_imbalance_comparison.csv')
imb_table

scale_pos_weight (train): 5.4603 | per fold train: [5.4644, 5.4593, 5.4593, 5.4593, 5.4593]


,MCC,Macro F1,F1-Score,Precision,TPR (Recall),TNR,auPR,auROC,Accuracy
I0 Tanpa treatment,0.6045 ± 0.0280,0.8010 ± 0.0141,0.6597 ± 0.0242,0.7091 ± 0.0248,0.6168 ± 0.0250,0.9537 ± 0.0042,0.7433 ± 0.0267,0.9302 ± 0.0088,0.9015 ± 0.0068
I1 Class weighting (spw),0.6137 ± 0.0232,0.7943 ± 0.0121,0.6692 ± 0.0187,0.5539 ± 0.0196,0.8458 ± 0.0229,0.8750 ± 0.0090,0.7401 ± 0.0263,0.9302 ± 0.0080,0.8705 ± 0.0089
I2 SMOTE 1:1 (= D0),0.6257 ± 0.0228,0.8110 ± 0.0115,0.6857 ± 0.0190,0.6343 ± 0.0217,0.7470 ± 0.0276,0.9209 ± 0.0075,0.7307 ± 0.0233,0.9265 ± 0.0076,0.8940 ± 0.0069


In [9]:
# Selisih per fold terhadap baseline (fold identik, jadi bisa dibandingkan berpasangan)
base = imb_folds['I2 SMOTE 1:1 (= D0)']
rows = []
for name in ['I0 Tanpa treatment', 'I1 Class weighting (spw)']:
    for m in ['MCC', 'Macro F1', 'auPR']:
        diff = imb_folds[name][m] - base[m]
        rows.append({'Varian': name, 'Metrik': m, 'Mean selisih vs D0': round(diff.mean(), 4),
                     'Std selisih': round(diff.std(ddof=0), 4),
                     'Fold lebih baik dari D0': f'{(diff > 0).sum()}/5'})
paired = pd.DataFrame(rows)
paired.to_csv(f'{TABLE_DIR}/tugas4_imbalance_paired_diff.csv', index=False)
paired

,Varian,Metrik,Mean selisih vs D0,Std selisih,Fold lebih baik dari D0
0,I0 Tanpa treatment,MCC,-0.0212,0.0145,1/5
1,I0 Tanpa treatment,Macro F1,-0.0099,0.0077,1/5
2,I0 Tanpa treatment,auPR,0.0125,0.0169,3/5
3,I1 Class weighting (spw),MCC,-0.0120,0.0219,2/5
4,I1 Class weighting (spw),Macro F1,-0.0166,0.0113,0/5
5,I1 Class weighting (spw),auPR,0.0094,0.0092,4/5


### Interpretasi Step 3

- **SMOTE 1:1 (D0) memberi MCC dan Macro F1 tertinggi.** MCC 0,6257 vs 0,6137 (class weighting) dan
  0,6045 (tanpa treatment). Secara berpasangan per fold, SMOTE lebih baik dari tanpa treatment di 4/5 fold
  (selisih rata-rata 0,0212, sekitar satu std). Selisih terhadap class weighting (0,0120) lebih kecil dari
  std antar fold, jadi keunggulannya atas class weighting belum bisa disebut meyakinkan.
- **Treatment menggeser trade-off precision/recall.** Tanpa treatment model konservatif (Precision 0,71,
  Recall 0,62). `scale_pos_weight` 5,46 menggeser terlalu jauh (Recall 0,85, Precision 0,55, TNR turun ke 0,875).
  SMOTE berada di tengah, dan titik tengah inilah yang paling disukai MCC.
- **auPR justru tertinggi tanpa treatment** (0,7433 vs 0,7307 untuk SMOTE). Artinya kemampuan model
  *mengurutkan* sesi berdasarkan peluang beli tidak membaik karena balancing; yang berubah terutama
  titik operasi di threshold 0,5. Keuntungan SMOTE pada MCC lebih tepat dibaca sebagai efek kalibrasi
  threshold, bukan karena model belajar pola yang lebih baik.

**Keputusan:** SMOTE 1:1 dipertahankan sebagai imbalance treatment terpilih untuk D1/D3, karena unggul di
primary metric (MCC) dan konsisten dengan baseline. Karena I2 identik dengan D0, eksperimen "D1 = data dengan
imbalance treatment" pada tabel Step 5 sama dengan D0; varian I0 dan I1 dilaporkan sebagai pembanding.